<div style="width:100vw; margin-left:-50vw; left:50%; position:relative;">
  <img src="https://i.postimg.cc/MTBpfsYw/CURSO-PYTHON-SIMPEG.png" style="width:100%;">
</div>

### 🎯 Reto principal

El objetivo de la práctica es construir una red neuronal inversa que estime el modelo de resistividades del subsuelo a partir de la respuesta magnetotelúrica:

$$
\boxed{
G_{\theta}:\; Z_{xy}\longrightarrow\hat{\rho}
}
$$

donde:

- $Z_{xy}$ corresponde a la respuesta magnetotelúrica de entrada.
- $G_{\theta}$ representa la red neuronal inversa.
- $\hat{\rho}$ corresponde a las **5 resistividades estimadas** por la red.
- $\rho$ corresponde a las **5 resistividades verdaderas** del modelo.

---

### 📦 Datos suministrados

Para desarrollar el reto se proporciona un dataset sintético compuesto por **250 000 modelos MT 1D**.

Para cada modelo se dispone de:

- **5 resistividades verdaderas**:

$$
\rho =
[\rho_1,\rho_2,\rho_3,\rho_4,\rho_5]
$$

- **4 espesores**, correspondientes a las primeras cuatro capas. La quinta capa se considera un semiespacio:

$$
h =
[h_1,h_2,h_3,h_4]
$$

- Una respuesta magnetotelúrica $Z_{xy}$ calculada para **31 frecuencias**:

$$
Z_{xy}=
[Z(f_1),Z(f_2),...,Z(f_{31})]
$$

Cada valor de $Z_{xy}$ contiene una **parte real y una parte imaginaria**.

- **31 frecuencias** distribuidas entre:

$$
10^{-3}\;Hz
\quad\text{y}\quad
10^{3}\;Hz
$$

Estas frecuencias son las mismas para todos los modelos.

- El **operador directo MT 1D** $F$, que permite calcular la respuesta magnetotelúrica a partir de las resistividades, espesores y frecuencias.

- Una versión **diferenciable del operador en PyTorch**, que permite incorporar el comportamiento físico dentro del entrenamiento de la red neuronal.

Por lo tanto, para cada modelo se tiene:

$$
\boxed{
5\text{ resistividades}
+
4\text{ espesores}
+
31\text{ frecuencias}
\longrightarrow
31\text{ respuestas }Z_{xy}
}
$$

---

### 🧠 Entrenamiento de la red

El entrenamiento deberá combinar tres términos: **supervisado, físico y regularización**.

---

#### 1. Pérdida supervisada

La componente supervisada evalúa qué tan cercana es la resistividad estimada por la red a la resistividad verdadera:

$$
\boxed{
\mathcal{L}_{sup}
=
\left\|
\rho-G_{\theta}(Z_{xy})
\right\|^2
}
$$

En otras palabras:

$$
\rho
\quad\longleftrightarrow\quad
\hat{\rho}
$$

Esta componente utiliza las resistividades verdaderas disponibles en el dataset.

---

#### 2. Pérdida física

La resistividad estimada por la CNN se introduce nuevamente en el **operador directo MT 1D**:

$$
\hat{Z}_{xy}
=
F(\hat{\rho})
=
F(G_{\theta}(Z_{xy}))
$$

Posteriormente se compara la respuesta magnetotelúrica reconstruida con la respuesta original:

$$
\boxed{
\mathcal{L}_{phys}
=
\left\|
Z_{xy}
-
F(G_{\theta}(Z_{xy}))
\right\|
}
$$

Por lo tanto, el flujo físico es:

$$
Z_{xy}
\longrightarrow
G_{\theta}
\longrightarrow
\hat{\rho}
\longrightarrow
F
\longrightarrow
\hat{Z}_{xy}
$$

y se busca que:

$$
\hat{Z}_{xy}
\approx
Z_{xy}
$$

En esta práctica, la comparación física se realiza para las **31 frecuencias**, utilizando la amplitud y la fase de la impedancia.

---

#### 3. Regularización

También se incluye un término de regularización para penalizar cambios muy fuertes entre las resistividades de capas consecutivas:

$$
\boxed{
\mathcal{L}_{reg}
=
\frac{1}{4}
\sum_{j=1}^{4}
\left(
\hat{\rho}_{j+1}
-
\hat{\rho}_{j}
\right)^2
}
$$

Este término favorece modelos de resistividad con transiciones menos abruptas entre capas.

---

### 🧩 Función de pérdida total

Finalmente, el entrenamiento debe integrar los tres términos:

$$
\boxed{
\mathcal{L}_{total}
=
\lambda_{sup}\mathcal{L}_{sup}
+
\lambda_{phys}\mathcal{L}_{phys}
+
\lambda_{reg}\mathcal{L}_{reg}
}
$$

Esta expresión corresponde al esquema de aprendizaje **supervisado y guiado por la física** que se busca implementar durante el reto.

Los valores utilizados inicialmente son:

$$
\lambda_{sup}=1.0
$$

$$
\lambda_{phys}=0.2
$$

$$
\lambda_{reg}=10^{-3}
$$

Por lo tanto, la red deberá aprender simultáneamente a:

- reproducir las **resistividades verdaderas**;
- generar modelos cuya respuesta física reproduzca el $Z_{xy}$ de entrada;
- mantener una regularización entre las capas estimadas.

> **Importante:** los archivos del dataset se entregan ya generados. La sección de generación se incluye únicamente para comprender y reproducir el procedimiento si se desea.

## 🗺️ Esquema general de la práctica

La siguiente figura resume la relación entre los datos MT, la CNN inversa y el operador físico diferenciable.
Arquitectura de la red y flujo de entrenamiento:<div style="width:100vw; margin-left:-50vw; left:50%; position:relative;">
  <img src="https://i.postimg.cc/Gtp2DxFW/Chat-GPT-Image-11-ago-2026-11-14-58-a-m.png" style="width:100%;">
</div>)

---

## 1. El problema directo y el problema inverso

En MT 1D, el subsuelo se representa mediante un modelo estratificado:

$$
\mathbf{m}
=
\left[
\rho_1,\rho_2,\ldots,\rho_N;
h_1,h_2,\ldots,h_{N-1}
\right]
$$

donde:

- $\rho_j$ es la resistividad eléctrica de la capa $j$;
- $h_j$ es su espesor;
- la última capa se considera un **semiespacio**, por lo que no requiere espesor.

### Problema directo

A partir del modelo $\mathbf{m}$ se calcula la respuesta MT:

$$
\boxed{
\mathbf{d}=F(\mathbf{m})
}
$$

En este cuadernillo:

$$
\mathbf{d}=Z_{xy}
$$

por lo tanto:

$$
\boxed{
Z_{xy}=F(\rho,h)
}
$$

### Problema inverso

El objetivo de la red es aproximar la operación contraria:

$$
\boxed{
\hat{\rho}=G_{\theta}(Z_{xy})
}
$$

La CNN no conoce explícitamente la función inversa de $F$; aprende esta relación a partir de los ejemplos suministrados.

---

## 2. Dataset utilizado en la práctica

Se trabajará con **250 000 modelos sintéticos**.

### Configuración

| Parámetro | Valor |
|---|---:|
| Número de modelos | **250 000** |
| Capas de resistividad | **5** |
| Espesores por modelo | **4** |
| Profundidad máxima | **10 000 m** |
| Número de frecuencias | **31** |
| Rango de frecuencia | $10^{-3}$ a $10^{3}$ Hz |
| Rango aproximado de resistividad | $1$ a $1000\;\Omega\cdot m$ |

Las frecuencias se distribuyen logarítmicamente:

$$
f=
\operatorname{logspace}(-3,3,31)
$$

y las resistividades se generan de forma aleatoria en escala logarítmica.

### Archivos que se suministran

| Archivo | Información | Forma |
|---|---|---|
| `resistivities_250k.npy` | resistividades verdaderas | `(250000, 5)` |
| `thicknesses_250k.npy` | espesores | `(250000, 4)` |
| `Zxy_250k.npy` | parte real e imaginaria de $Z_{xy}$ | `(250000, 31, 2)` |
| `frequencies.npy` | frecuencias | `(31,)` |

La impedancia se almacena como:

$$
Z_{xy}
=
\operatorname{Re}(Z_{xy})
+
i\,\operatorname{Im}(Z_{xy})
$$

de modo que:

```text
Zxy[..., 0]  → parte real
Zxy[..., 1]  → parte imaginaria
```

---

## 3. Preparación del entorno

In [ ]:
#@title ▶️ Ejecutar importaciones
import os
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from scipy import constants
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

mu = constants.mu_0

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

Device: cpu
PyTorch: 2.11.0+cpu
NumPy: 2.0.2


---

## 4. Parámetros y ubicación de los archivos


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title ⚙️ Configuración

#@title ⚙️ Configuración

DATA_DIR = "/content/drive/MyDrive/Curso_07" #@param {type:"string"}

N_MODELS = 250_000
N_LAYERS = 5
N_FREQ = 31
DEPTH_MAX = 10000

RHO_PATH = os.path.join(DATA_DIR, "resistivities_250k.npy")
THICK_PATH = os.path.join(DATA_DIR, "thicknesses_250k.npy")
Z_PATH = os.path.join(DATA_DIR, "Zxy_250k.npy")
FREQ_PATH = os.path.join(DATA_DIR, "frequencies.npy")

frequencies = np.logspace(
    -3, 3, N_FREQ
).astype(np.float32)

print("📁 Carpeta de trabajo:")
print(os.path.abspath(DATA_DIR))

print("\n📄 Archivos encontrados:")

if os.path.exists(DATA_DIR):
    archivos = os.listdir(DATA_DIR)

    if len(archivos) == 0:
        print("La carpeta está vacía.")
    else:
        for archivo in archivos:
            print(" -", archivo)
else:
    print("❌ La ruta indicada no existe.")

📁 Carpeta de trabajo:
/content/drive/MyDrive/Curso_07

📄 Archivos encontrados:
 - frequencies.npy
 - resistivities_1M.npy
 - thicknesses_1M.npy
 - Zxy_1M.npy


---

# Parte A · ¿Cómo se generaron los datos?

Las siguientes secciones permiten comprender la construcción del dataset.  
**No es necesario volver a generar los 250 000 modelos durante la práctica.**

## 5. Operador directo MT 1D

Para cada frecuencia:

$$
\omega=2\pi f
$$

La impedancia de la capa inferior o semiespacio se calcula como:

$$
Z_N=
\sqrt{i\omega\mu_0\rho_N}
$$

A partir de allí se realiza una recursión desde la profundidad hacia la superficie.

Para una capa $j$:

$$
\gamma_j=
\sqrt{\frac{i\omega\mu_0}{\rho_j}}
$$

$$
w_j=\rho_j\gamma_j
$$

El efecto del espesor está dado por:

$$
e_j=
\exp(-2h_j\gamma_j)
$$

y el contraste con el medio inferior mediante:

$$
r_j=
\frac{w_j-Z_{j+1}}
{w_j+Z_{j+1}}
$$

Finalmente:

$$
\boxed{
Z_j=
w_j
\frac{1-r_je_j}
{1+r_je_j}
}
$$

Al alcanzar la primera capa se obtiene la impedancia observada en superficie:

$$
Z_{xy}(f)=Z_1
$$

In [ ]:
#@title 📐 Ver operador directo NumPy
def MTforwardModel(resistivities, thicknesses, frequencies):
    """Operador directo MT 1D en NumPy."""
    n = len(resistivities)
    Zxy = []

    for frequency in frequencies:
        omega = 2.0 * math.pi * frequency
        impedances = [0j] * n

        # Semiespacio inferior
        impedances[-1] = np.sqrt(
            omega * mu * resistivities[-1] * 1j
        )

        # Recursión hacia la superficie
        for j in range(n - 2, -1, -1):
            rho_j = resistivities[j]
            h_j = thicknesses[j]

            gamma_j = np.sqrt(
                (omega * mu / rho_j) * 1j
            )

            wj = gamma_j * rho_j
            ej = np.exp(-2.0 * h_j * gamma_j)

            Z_below = impedances[j + 1]

            rj = (
                (wj - Z_below) /
                (wj + Z_below)
            )

            re = rj * ej

            impedances[j] = (
                wj *
                ((1.0 - re) / (1.0 + re))
            )

        Zxy.append(impedances[0])

    return np.asarray(Zxy)

## 6. Generación de los modelos de resistividad

Cada modelo contiene cinco resistividades:

$$
\rho=
[\rho_1,\rho_2,\rho_3,\rho_4,\rho_5]
$$

y cuatro espesores:

$$
h=
[h_1,h_2,h_3,h_4]
$$

La quinta capa corresponde al semiespacio.

Las resistividades se generan aleatoriamente en escala logarítmica:

$$
\log_{10}(\rho)\sim U(0,3)
$$

por lo que aproximadamente:

$$
1\leq\rho\leq1000\;\Omega\cdot m
$$

In [ ]:
#@title 🧱 Ver generador de modelos
def randomModel(n=5, depthmax=10000, res=(0, 3)):
    """
    Generador de modelos sintéticos 1D.

    Basado en el generador original de Paul Goyes.
    """
    indx_rho = np.random.choice(
        200, n, replace=True
    )

    rho_pool = np.random.uniform(
        res[0], res[1], 200
    )

    resistivities_log = rho_pool[indx_rho]

    dz = depthmax // (n - 1)
    z = np.arange(n) * dz
    thicknesses = np.diff(z)

    return (
        10 ** resistivities_log,
        thicknesses
    )

## 7. Generación de los 250 000 modelos

Para cada modelo se realiza:

$$
(\rho,h)
\longrightarrow
F(\rho,h)
\longrightarrow
Z_{xy}
$$

La información se escribe directamente en disco mediante `open_memmap`.

> ⚠️ Esta celda está **desactivada por defecto** porque los archivos ya se entregan generados.

In [ ]:
#@title 💾 Generar dataset (opcional) — No es necesario ejecutarlo si ya tienes los archivos generados
GENERAR_DATASET = False  #@param {type:"boolean"}

if not GENERAR_DATASET:
    print(
        "Dataset ya suministrado: "
        "no se volverán a generar los 250 000 modelos."
    )
else:
    np.random.seed(42)
    os.makedirs(DATA_DIR, exist_ok=True)

    rho_mem = np.lib.format.open_memmap(
        RHO_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(N_MODELS, N_LAYERS)
    )

    thick_mem = np.lib.format.open_memmap(
        THICK_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(N_MODELS, N_LAYERS - 1)
    )

    Z_mem = np.lib.format.open_memmap(
        Z_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(N_MODELS, N_FREQ, 2)
    )

    start = time.time()

    for i in tqdm(
        range(N_MODELS),
        desc="Generando dataset MT"
    ):
        rho_i, h_i = randomModel(
            n=N_LAYERS,
            depthmax=DEPTH_MAX
        )

        Z_i = MTforwardModel(
            rho_i,
            h_i,
            frequencies
        )

        rho_mem[i] = rho_i.astype(np.float32)
        thick_mem[i] = h_i.astype(np.float32)

        Z_mem[i, :, 0] = np.real(Z_i)
        Z_mem[i, :, 1] = np.imag(Z_i)

    np.save(FREQ_PATH, frequencies)

    rho_mem.flush()
    thick_mem.flush()
    Z_mem.flush()

    elapsed = (time.time() - start) / 60

    print(
        f"Dataset generado en {elapsed:.2f} min"
    )

---

# Parte B · Operador físico diferenciable

## 8. ¿Por qué debe ser diferenciable?

La CNN estima:

$$
\hat{\rho}=G_{\theta}(Z_{xy})
$$

Para incorporar física en el entrenamiento, la resistividad estimada vuelve a pasar por el operador directo:

$$
\hat{Z}_{xy}=F(\hat{\rho})
$$

y se compara con la respuesta original.

Para poder actualizar los pesos $\theta$ mediante retropropagación, PyTorch debe poder calcular:

$$
\frac{\partial\mathcal{L}}
{\partial\hat{\rho}}
$$

y posteriormente:

$$
\frac{\partial\mathcal{L}}
{\partial\theta}
$$

Por esta razón se suministra una versión del operador implementada completamente con operaciones diferenciables de PyTorch.

In [ ]:
#@title 🧠 Ver operador diferenciable
MU0 = 4.0 * math.pi * 1e-7

def MTforwardModel_torch(
    resistivities,
    thicknesses,
    frequencies
):
    """
    Operador directo MT 1D diferenciable.

    resistivities : (B, 5)
    thicknesses   : (B, 4) o (4,)
    frequencies   : (31,)

    retorna:
        Zxy complejo (B, 31)
    """
    real_dtype = resistivities.dtype
    dev = resistivities.device

    complex_dtype = (
        torch.complex64
        if real_dtype == torch.float32
        else torch.complex128
    )

    frequencies = frequencies.to(
        device=dev,
        dtype=real_dtype
    )

    if thicknesses.ndim == 1:
        thicknesses = thicknesses[None, :].repeat(
            resistivities.shape[0],
            1
        )

    thicknesses = thicknesses.to(
        device=dev,
        dtype=real_dtype
    )

    _, n_layers = resistivities.shape

    omega = 2.0 * math.pi * frequencies

    iwmu = torch.complex(
        torch.zeros_like(omega),
        MU0 * omega
    )[None, :].to(
        dtype=complex_dtype,
        device=dev
    )

    # Semiespacio
    rho_last = resistivities[:, -1][:, None].to(
        complex_dtype
    )

    Z_below = torch.sqrt(
        iwmu * rho_last
    )

    # Recursión
    for j in range(n_layers - 2, -1, -1):
        rho_j = resistivities[:, j][:, None].to(
            complex_dtype
        )

        h_j = thicknesses[:, j][:, None].to(
            complex_dtype
        )

        gamma_j = torch.sqrt(
            iwmu / rho_j
        )

        wj = gamma_j * rho_j
        ej = torch.exp(
            -2.0 * h_j * gamma_j
        )

        rj = (
            (wj - Z_below) /
            (wj + Z_below)
        )

        re = rj * ej

        Z_below = (
            wj *
            ((1.0 - re) / (1.0 + re))
        )

    return Z_below